In [1]:
import pandas as pd
import numpy as np



In [2]:
# 1. Base user profiles
users = pd.DataFrame({
    'user_id': [101, 102, 103],
    'signup_date': pd.to_datetime(['2026-01-05', '2026-02-10', '2026-01-15'])
})

# 2. Raw login event logs
events_data = [
    # User 101: Active Jan-Mar, stops in April (Churns in May)
    (101, '2026-01-10'), (101, '2026-01-20'), (101, '2026-02-05'), 
    (101, '2026-02-25'), (101, '2026-03-12'), (101, '2026-03-28'),
    # User 102: Active Feb-May (Retained)
    (102, '2026-02-15'), (102, '2026-03-01'), (102, '2026-04-10'), 
    (102, '2026-04-25'), (102, '2026-05-05'), (102, '2026-05-20'),
    # User 103: Active Jan, disappears in Feb (Churns in March)
    (103, '2026-01-18'), (103, '2026-01-25')
]
events = pd.DataFrame(events_data, columns=['user_id', 'timestamp'])
events['timestamp'] = pd.to_datetime(events['timestamp'])

In [3]:
users

,user_id,signup_date
0,101,2026-01-05
1,102,2026-02-10
2,103,2026-01-15


In [4]:
events

,user_id,timestamp
0,101,2026-01-10
1,101,2026-01-20
2,101,2026-02-05
3,101,2026-02-25
4,101,2026-03-12
5,101,2026-03-28
6,102,2026-02-15
7,102,2026-03-01
8,102,2026-04-10
9,102,2026-04-25


In [9]:
events.groupby("user_id")["timestamp"].agg(['min', 'max'])

,min,max
user_id,,
101,2026-01-10,2026-03-28
102,2026-02-15,2026-05-20
103,2026-01-18,2026-01-25


# Option 1: One Row Per Customer Per Cutoff Date (Multiple Snapshots)

* In this setup, your final dataset will have multiple rows for the same customer, but each row represents that customer at a different snapshot in time
* This approach loops through multiple months, aggregates historical data up to each month-end, and stacks them together.
* Option 1 contains multiple rows for users 101 and 103, showing how their profile data evolved dynamically through the months.

In [5]:
def make_snapshot(events_df, cutoff_date):
    cutoff = pd.to_datetime(cutoff_date)
    label_end = cutoff + pd.Timedelta(days=30)
    
    # Features: Everything BEFORE cutoff
    hist = events_df[events_df['timestamp'] <= cutoff]
    features = hist.groupby('user_id').size().to_frame('total_logins').reset_index()
    
    # Labels: Active in the 30 days AFTER cutoff?
    future = events_df[(events_df['timestamp'] > cutoff) & (events_df['timestamp'] <= label_end)]
    active_future = future['user_id'].unique()
    
    features['cutoff_date'] = cutoff
    features['churn_next_30d'] = features['user_id'].apply(lambda x: 0 if x in active_future else 1)
    return features



In [8]:
# Generate snapshots for multiple historical cutoff dates
snapshots = [make_snapshot(events, d) for d in ['2026-01-31', '2026-02-28', '2026-03-31']]
df_option1 = pd.concat(snapshots, ignore_index=True)

print("--- OPTION 1: MULTIPLE SNAPSHOTS ---")
print(df_option1)

--- OPTION 1: MULTIPLE SNAPSHOTS ---
   user_id  total_logins cutoff_date  churn_next_30d
0      101             2  2026-01-31               0
1      103             2  2026-01-31               1
2      101             4  2026-02-28               0
3      102             1  2026-02-28               0
4      103             2  2026-02-28               1
5      101             6  2026-03-31               1
6      102             2  2026-03-31               0
7      103             2  2026-03-31               1


# Option 2: Strictly One Row Per Customer (Single Snapshot)

This approach freezes time on one single calendar day for the entire user base. It perfectly reflects how a live production model calculates risk scores.

In [10]:
# Choose a single point in time
SINGLE_CUTOFF = pd.to_datetime('2026-02-28')
LABEL_WINDOW_END = SINGLE_CUTOFF + pd.Timedelta(days=30)

# Features: All history up to Feb 28
features_op2 = events[events['timestamp'] <= SINGLE_CUTOFF].groupby('user_id').size().to_frame('total_logins').reset_index()

# Labels: Did they login during March?
future_op2 = events[(events['timestamp'] > SINGLE_CUTOFF) & (events['timestamp'] <= LABEL_WINDOW_END)]
active_march = future_op2['user_id'].unique()

features_op2['churn_in_march'] = features_op2['user_id'].apply(lambda x: 0 if x in active_march else 1)

print("\n--- OPTION 2: SINGLE SNAPSHOT ---")
print(features_op2)


--- OPTION 2: SINGLE SNAPSHOT ---
   user_id  total_logins  churn_in_march
0      101             4               0
1      102             1               0
2      103             2               1


# Option 3: Sequence / Time-Series Format

* This approach pivots the data so that monthly behavioral volumes are mapped to independent columns across a single row per user.
* Instead of flattening a customer's history into aggregate metrics (like total_logins), you keep each month's behavior as a separate column or sequential step in a single row.

In [11]:
# Extract month name or number from the event logs
events['month'] = events['timestamp'].dt.strftime('%b').str.lower()

# Pivot into month-by-month sequences
sequence_df = events.pivot_table(
    index='user_id', 
    columns='month', 
    values='timestamp', 
    aggfunc='count', 
    fill_value=0
).reset_index()

# Enforce explicit chronological column layout
feature_cols = ['jan', 'feb', 'mar', 'apr']
df_option3 = sequence_df[['user_id'] + feature_cols].copy()

# Label: Did they log in during May?
df_option3['churn_in_may'] = sequence_df['may'].apply(lambda x: 1 if x == 0 else 0)

print("\n--- OPTION 3: SEQUENCE FORMAT ---")
print(df_option3)



--- OPTION 3: SEQUENCE FORMAT ---
month  user_id  jan  feb  mar  apr  churn_in_may
0          101    2    2    2    0             1
1          102    0    1    1    2             0
2          103    2    0    0    0             1


# Option 4: Relative "Tenure-Based" Cutoffs

This approach shifts focus away from calendar boundaries to instead measure user milestones relative to their personal sign-up day.

In [12]:
# Merge profile data to access individual sign-up timelines
merged = events.merge(users, on='user_id')

# Calculate the precise age of the user at the exact moment of each log event
merged['days_since_signup'] = (merged['timestamp'] - merged['signup_date']).dt.days

# Features: Total activity logged within the first 30 days of lifecycle
features_op4 = merged[merged['days_since_signup'] <= 30].groupby('user_id').size().to_frame('logins_first_30_days').reset_index()

# Labels: Did they maintain activity between lifecycle day 31 and day 60?
future_op4 = merged[(merged['days_since_signup'] > 30) & (merged['days_since_signup'] <= 60)]
active_lifecycle_window = future_op4['user_id'].unique()

features_op4['churn_day_31_to_60'] = features_op4['user_id'].apply(lambda x: 0 if x in active_lifecycle_window else 1)

print("\n--- OPTION 4: TENURE-BASED ---")
print(features_op4)



--- OPTION 4: TENURE-BASED ---
   user_id  logins_first_30_days  churn_day_31_to_60
0      101                     2                   0
1      102                     2                   0
2      103                     2                   1


In [27]:
data = [1, "loft", 2, 0.5, "Jan"], [1, "loft", 3, 0.8, "Feb"]
data

([1, 'loft', 2, 0.5, 'Jan'], [1, 'loft', 3, 0.8, 'Feb'])

In [28]:
df = pd.DataFrame(data, columns=["id", "brand", "score", "prob", "as_of_dt"])
df

,id,brand,score,prob,as_of_dt
0,1,loft,2,0.5,Jan
1,1,loft,3,0.8,Feb


In [29]:
df.pivot(index=["id","brand"], columns="as_of_dt", values=["score", "prob"])

score      prob     
as_of_dt   Feb  Jan  Feb  Jan
id brand                     
1  loft    3.0  2.0  0.8  0.5